# Example 1: Water Fitting
In this example, we will show how to perform a neural network fit of a small water dataset. We will load the data, apply physical transformations, train a neural network, and evaluate its performance.

First, we must load our raw dataset. The data is handled by a `Dataset` class, which extracts the features and target variables from the CSV file.

In [8]:
from pes_nn_fit.dataset import Dataset

data_file = "../../data/water.csv"

# Load dataset
dataset = Dataset(file_location=data_file, file_type='pes-learn')
print("Features (first 5):\n", dataset.data[:5])
print("Targets (first 5):\n", dataset.target[:5])

Features (first 5):
 [[ 0.8  0.8 40. ]
 [ 0.8  0.8 50. ]
 [ 0.8  0.8 60. ]
 [ 0.8  0.8 70. ]
 [ 0.8  0.8 80. ]]
Targets (first 5):
 [[-76.03292207]
 [-76.12224131]
 [-76.17668111]
 [-76.21368439]
 [-76.23944761]]


Next, we explicitly define and apply our preprocessing steps. For water (`h2o`), we will apply geometric canonicalization, a Morse transform, and a Permutationally Invariant Polynomial (PIP) transform. Finally, both the data and the target will be standard-scaled. Note that the input of most classes is a dictionary. 

In [9]:
from pes_nn_fit.preprocess import Preprocessor

# In input dict specify the parameters of the preprocessing
pre_cfg = {
    "canonicalize_geom": True,
    "molecule": "h2o",
    "morse_transform": True,
    "morse_param": 1.0,
    "pip_transform": True,
    "scaler_data": "std",
    "scaler_target": "std"
}

# Initialize and fit the preprocessor
preprocessor = Preprocessor(cfg=pre_cfg)
preprocessor.fit(dataset.data, dataset.target)

# Transform the full dataset
X_trans, y_trans = preprocessor.transform(dataset.data, dataset.target)

# Wrap the transformed arrays back into a Dataset object
transformed_dataset = Dataset(array_tuple=(X_trans, y_trans))

Now we split the transformed dataset into training and testing sets to evaluate our model later.

In [10]:
# Split 80% for training and 20% for testing (validation set is unnecessary here)
trainset, testset = transformed_dataset.split_random(train_size=0.8, val_size=0.0, test_size=0.2)

print(f"Training samples: {trainset.n_samples}")
print(f"Testing samples: {testset.n_samples}")

Training samples: 792
Testing samples: 198


We can now initialize our neural network. The `Model` class takes the input size (dynamically calculated from our preprocessed features) and the network architecture parameters.

In [11]:
from pes_nn_fit.model import Model

model_hyperparams = {
    "layers": "8,8,8",
    "activation": "tanh"
}

input_size = trainset.n_features 
model = Model(input_size=input_size, hyperparams=model_hyperparams)

To optimize the network, we configure the `Trainer`. We will use the Levenberg-Marquardt (LM) optimizer, which is highly effective for fitting potential energy surfaces.

In [17]:
import torch

from pes_nn_fit.model import Trainer

trainer_hyperparams = {
    "early_stopping": {
        "enabled": False
    },
    "lm": {
        "lr": 0.1,
        "max_epoch": 100,
        "attempts_per_step": 10,
        "use_vmap": True,
        "batch_size": 1000,
        "solve_method": "qr"
    }
}

# Initialize trainer with torch.float64 for high precision
trainer = Trainer(hyperparams=trainer_hyperparams, dtype=torch.float64)

# Cast the model to float64 to match the Trainer and dataset
model = model.to(torch.float64)

# Run the training loop
result = trainer.train(
    model=model,
    optimizer_type="lm",
    trainset=trainset
)

Finally, we evaluate our trained model on the unseen test set. We predict the scaled energy values, inverse-transform them back to their original physical units, and compute the Root Mean Square Error (RMSE).

In [18]:
import numpy as np

# Set model to evaluation mode
model.eval()

# Convert test features to tensor
X_test_tensor = torch.tensor(testset.data, dtype=torch.float64)

# Make predictions
with torch.no_grad():
    y_pred_scaled = model(X_test_tensor).numpy()

# Inverse transform to get back to the original physical scale (e.g., Hartrees)
y_pred = preprocessor.inverse_transform_y(y_pred_scaled)
y_actual = preprocessor.inverse_transform_y(testset.target)

# Calculate RMSE
rmse = np.sqrt(np.mean((y_pred - y_actual) ** 2))
print(f"Test RMSE: {rmse:.6e}")

Test RMSE: 3.987329e-02
